# 02 — ERA5-Land rainfall for the storm events

Accumulated rainfall over 3-, 5- and 7-day windows centred on each event's
closest approach, for the storms found in `01_windspeed_ibtracs.ipynb`.

**Run second.** Reads `outputs/ibtracs_impact_storms_*.csv`, writes
`outputs/{ISO3}_rainfall_era5_*.csv`.

`total_precipitation_hourly` is an hourly **rate** in metres — no de-accumulation
needed, ×1000 for mm h⁻¹.

> **Small-island caveat.** At ~9 km and masked to land, a small island is a
> handful of cells. The area mean smooths away the orographic maxima that drive
> flash flooding: an island-scale index, not point rainfall.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Repo root on sys.path, whether launched from notebooks/ or the repo root.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(ROOT))

import time

import ee
import numpy as np
import pandas as pd

import config
from storm_utils import load_boundary
from plot_utils import (
    cat_color, normalise_cat, set_style,
    plot_stacked_event_bars, plot_event_bars, plot_distributions,
    plot_rainfall_vs_category, plot_rainfall_vs, plot_exceedance,
    plot_heatmap, plot_trend_over_time,
)

config.ensure_dirs()
set_style()
print(config.summary())

In [ ]:
if not config.GEE_PROJECT:
    raise RuntimeError("EE_PROJECT is not set — export it, or set GEE_PROJECT in config.py")

try:
    ee.Initialize(project=config.GEE_PROJECT)
except Exception:
    ee.Authenticate()          # only prompts if the cached token is missing/expired
    ee.Initialize(project=config.GEE_PROJECT)

print(f"[OK] Earth Engine initialised on project '{config.GEE_PROJECT}'")

## 2. Study-area geometry

The country polygon, buffered by `config.GEOMETRY_BUFFER_DEG` (~5 km) so a small
island always intersects at least one ERA5-Land cell, without pulling in a
neighbouring territory.

In [ ]:
gdf, _ = load_boundary(config.ISO3)
geom_ll = gdf.to_crs("EPSG:4326").union_all()
GEOMETRY = ee.Geometry(geom_ll.buffer(config.GEOMETRY_BUFFER_DEG).__geo_interface__,
                       proj="EPSG:4326", geodesic=False)

print(f"[OK] Geometry for {config.COUNTRY_NAME} ({config.ISO3})")
print(f"     Bounding box: {[round(float(b), 3) for b in gdf.total_bounds]}")

# Cells over the geometry — decides whether grid-max is a second metric at all.
first = ee.ImageCollection(config.ERA5_COLLECTION).first().select(config.ERA5_BAND)
n_px = first.reduceRegion(ee.Reducer.count(), GEOMETRY,
                          config.ERA5_SCALE_M, maxPixels=1e9).getInfo()
n_cells = list(n_px.values())[0]
print(f"     ERA5-Land cells over the geometry: {n_cells}")
if n_cells <= 2:
    print("     [warn] area mean and grid max are effectively the same number")

## 3. Extraction

Each window is **centred** on closest approach — a 3-day total spans 36 h either
side. That is the event's total water, not a forecast-window accumulation, so a
trigger calibrated here must be re-expressed in forecast terms before use.
Windows are defined in `config.WINDOW_CONFIGS`.

In [ ]:
def extract_rainfall_window(start_dt, end_dt, geom, label, compute_peak=False):
    """Area-mean rainfall total (mm) for one window; for `compute_peak` windows
    also the grid-cell maximum total and the mean per-cell peak hourly rate.

    Mean and max share one combined reducer: 1 getInfo per window (2 with peak).
    """
    fmt = "%Y-%m-%dT%H:%M:%S"
    to_mm = lambda v: round(v * 1000, 2) if v is not None else np.nan

    result = {f"rain_{label}_mm": np.nan}
    if compute_peak:
        result |= {f"rain_{label}_gridmax_mm": np.nan, f"peak_{label}_mmph": np.nan}

    col = (
        ee.ImageCollection(config.ERA5_COLLECTION)
        .filterDate(start_dt.strftime(fmt), end_dt.strftime(fmt))
        .filterBounds(geom)
        .select(config.ERA5_BAND)
    )

    reducer = ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True)
    totals = (
        col.sum()
        .reduceRegion(reducer, geom, config.ERA5_SCALE_M, maxPixels=1e9, bestEffort=True)
        .getInfo()
    )
    # An empty collection reduces to None rather than raising.
    result[f"rain_{label}_mm"] = to_mm(totals.get(f"{config.ERA5_BAND}_mean"))

    if compute_peak:
        result[f"rain_{label}_gridmax_mm"] = to_mm(totals.get(f"{config.ERA5_BAND}_max"))
        peak = (
            col.max()
            .reduceRegion(ee.Reducer.mean(), geom, config.ERA5_SCALE_M,
                          maxPixels=1e9, bestEffort=True)
            .getInfo()
        )
        result[f"peak_{label}_mmph"] = to_mm(peak.get(config.ERA5_BAND))

    return result


def as_utc(value):
    """Coerce a timestamp to UTC whether or not it is already tz-aware."""
    ts = pd.Timestamp(value)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")

### Event list

The IBTrACS impact set, optionally extended with a national met-service event
list at `config.MET_EVENTS_CSV` (columns: `event_id`, `event_datetime`, `name`,
and optionally `ss_category`, `obs_rain_mm`). The analysis runs on IBTrACS alone
if that file is absent.

In [ ]:
KEEP = ["name", "ss_category", "max_wind_kmh", "closest_dist_km", "source", "obs_rain_mm"]

storms = pd.read_csv(config.STORMS_CSV, parse_dates=["closest_approach_date"])
storms["event_datetime"] = storms["closest_approach_date"]
storms["source"] = "ibtracs"

HAS_MET = config.MET_EVENTS_CSV.exists()
if HAS_MET:
    met = pd.read_csv(config.MET_EVENTS_CSV, parse_dates=["event_datetime"])
    met["source"] = met.get("source", "met")
    events = pd.concat([storms, met], ignore_index=True)
else:
    met = None
    events = storms.copy()
    print(f"[info] No {config.MET_EVENTS_CSV.name} — IBTrACS events only.")

for col in KEEP:                       # keep the schema stable either way
    if col not in events.columns:
        events[col] = np.nan

events = events.sort_values("event_datetime").reset_index(drop=True)

# A met event may already be in the IBTrACS set — check before trusting the count.
if HAS_MET:
    for _, r in met.iterrows():
        d = (storms["event_datetime"] - r["event_datetime"]).abs()
        if d.min() <= pd.Timedelta(days=3):
            print(f"[dup?] {r.get('event_id', r['name'])} ~ {storms.loc[d.idxmin(), 'name']}")

print(f"{len(storms)} IBTrACS + {0 if met is None else len(met)} met = {len(events)} events")

In [ ]:
# Sanity-check one event before committing to the full run
probe = events.loc[events["max_wind_kmh"].idxmax()]
probe_ca = as_utc(probe["event_datetime"])
print(f"Probe: {probe['name']} | {probe_ca} | {probe['max_wind_kmh']:.0f} km/h")

for label, (total_hours, compute_peak) in config.WINDOW_CONFIGS.items():
    half = pd.Timedelta(hours=total_hours / 2)
    print(f"  {label}: {extract_rainfall_window(probe_ca - half, probe_ca + half, GEOMETRY, label, compute_peak)}")

In [ ]:
# Full extraction (one getInfo per window per event — expect a few minutes)
records = []

for i, (_, row) in enumerate(events.iterrows(), start=1):
    ca = as_utc(row["event_datetime"])
    print(f"[{i}/{len(events)}] {row['name']} | {ca} | {row['ss_category']}")

    record = {k: row[k] for k in KEEP} | {"year": ca.year, "event_datetime": ca}
    for label, (total_hours, compute_peak) in config.WINDOW_CONFIGS.items():
        half = pd.Timedelta(hours=total_hours / 2)
        record |= extract_rainfall_window(ca - half, ca + half, GEOMETRY, label, compute_peak)
        time.sleep(0.25)       # stay under the getInfo rate limit

    print(f"    3d={record['rain_3d_mm']} | 5d={record['rain_5d_mm']} "
          f"| 7d={record['rain_7d_mm']} mm")
    records.append(record)

COLS = (["year", "event_datetime"] + KEEP
        + ["rain_3d_mm", "rain_5d_mm", "rain_7d_mm", "rain_7d_gridmax_mm", "peak_7d_mmph"])
df_rainfall = pd.DataFrame(records)[COLS]
df_rainfall.to_csv(config.RAINFALL_CSV, index=False)
print(f"\n[OK] {len(df_rainfall)} events → {config.RAINFALL_CSV}")

## 4. Analysis

**Resume here** if the extraction has already run — the cell below reloads the
CSV, so nothing downstream re-queries Earth Engine.

In [ ]:
df_rainfall = pd.read_csv(config.RAINFALL_CSV, parse_dates=["event_datetime"])
print(f"[OK] {len(df_rainfall)} events loaded")

In [ ]:
# Validation against gauge observations, where any exist (obs_rain_mm)
chk = df_rainfall[df_rainfall["obs_rain_mm"].notna()].copy()
if chk.empty:
    print("[info] No gauge observations to validate against.")
else:
    chk["ratio_3d"] = (chk["rain_3d_mm"] / chk["obs_rain_mm"]).round(2)
    chk["ratio_7d"] = (chk["rain_7d_mm"] / chk["obs_rain_mm"]).round(2)
    print(chk[["name", "obs_rain_mm", "rain_3d_mm", "rain_7d_mm",
               "peak_7d_mmph", "ratio_3d", "ratio_7d"]].to_string(index=False))
    print(f"\nMedian ERA5 / gauge — 3d: {chk['ratio_3d'].median():.2f}"
          f" | 7d: {chk['ratio_7d'].median():.2f}   (1.0 = agreement)")

# If this is ~1.0, reduceRegion is returning a single pixel and grid max is not
# a second metric.
ratio = (df_rainfall["rain_7d_gridmax_mm"] / df_rainfall["rain_7d_mm"]).median()
print(f"\nMedian grid max / area mean: {ratio:.2f}")

In [ ]:
# Shared arrays for the figures
df_plot = (
    df_rainfall
    .dropna(subset=["rain_3d_mm", "rain_5d_mm", "rain_7d_mm"])
    .sort_values("event_datetime")
    .reset_index(drop=True)
)
df_plot["cat"] = df_plot["ss_category"].apply(normalise_cat)


def short_label(name, max_len=14):
    """Trim long event names so the per-event axis stays readable."""
    name = str(name).title()
    return name if len(name) <= max_len else name[:max_len - 1] + "…"


CN, ISO3, OUT = config.COUNTRY_NAME, config.ISO3, config.OUTPUT_DIR

dates   = df_plot["event_datetime"].dt.strftime("%Y-%m").tolist()
years   = df_plot["event_datetime"].dt.year.to_numpy()
names   = [short_label(n) for n in df_plot["name"]]
cats    = df_plot["cat"].tolist()
r3      = df_plot["rain_3d_mm"].to_numpy()
r5      = df_plot["rain_5d_mm"].to_numpy()
r7      = df_plot["rain_7d_mm"].to_numpy()
gmax    = df_plot["rain_7d_gridmax_mm"].to_numpy()
peak    = df_plot["peak_7d_mmph"].to_numpy()
dist    = df_plot["closest_dist_km"].to_numpy()
wind    = df_plot["max_wind_kmh"].to_numpy()
colors  = [cat_color(c) for c in cats]
xlabels = [f"{d}\n{n}" for d, n in zip(dates, names)]

dropped = len(df_rainfall) - len(df_plot)
print(f"{len(df_plot)} events | categories: {sorted(set(cats))}"
      + (f" | {dropped} dropped for missing rainfall" if dropped else ""))

### Summary statistics

In [ ]:
metrics = {
    "3d total (mm)":      r3,
    "5d total (mm)":      r5,
    "7d total (mm)":      r7,
    "7d grid max (mm)":   gmax,
    "Peak hourly (mm/h)": peak,
}

rows = []
for lbl, arr in metrics.items():
    s = arr[~np.isnan(arr)]
    rows.append({
        "Metric": lbl, "N": len(s),
        "Mean": round(s.mean(), 1), "Median": round(np.median(s), 1),
        "Std": round(s.std(), 1), "Min": round(s.min(), 1),
        "P25": round(np.percentile(s, 25), 1),
        "P75": round(np.percentile(s, 75), 1),
        "P90": round(np.percentile(s, 90), 1),
        "P95": round(np.percentile(s, 95), 1),
        "Max": round(s.max(), 1),
    })

stats_df = pd.DataFrame(rows)
print(stats_df.to_string(index=False))

stats_path = OUT / f"{ISO3}_rainfall_era5_summary_statistics.csv"
stats_df.to_csv(stats_path, index=False)
print(f"\n  Saved → {stats_path}")

### Per-event rainfall

In [ ]:
plot_stacked_event_bars(r3, r5, r7, colors, xlabels, CN, cats, OUT,
                        f"2a_{ISO3}_rainfall_by_event.png")

plot_event_bars(gmax, colors, xlabels, "Rainfall grid max", "mm", CN, cats, OUT,
                f"2b_{ISO3}_rainfall_gridmax.png")

plot_event_bars(peak, colors, xlabels, "Mean peak hourly intensity", "mm/h", CN, cats, OUT,
                f"2c_{ISO3}_rainfall_peakintensity.png")

### Distributions and category relationship

In [ ]:
plot_distributions(r3, r5, r7, CN, OUT, f"3_{ISO3}_rainfall_distributions.png")

for arr, tag in [(r3, "3d"), (r5, "5d"), (r7, "7d")]:
    plot_rainfall_vs_category(arr, cats, CN, tag, OUT,
                              f"4_{ISO3}_rainfall_vs_category_{tag}.png")

### Rainfall vs proximity and intensity

If rainfall tracked wind category or closest approach closely, a wind-based
trigger would be enough. These panels are the evidence that it isn't.

In [ ]:
plot_rainfall_vs(dist, [r3, r5, r7], cats, ["3-day", "5-day", "7-day"],
                 "Closest approach distance (km)", CN, annotations=years,
                 output_dir=OUT, filename=f"5_{ISO3}_rainfall_vs_distance.png")

plot_rainfall_vs(wind, [r3, r5, r7], cats, ["3-day", "5-day", "7-day"],
                 "Maximum wind speed (km/h)", CN, annotations=years,
                 output_dir=OUT, filename=f"6_{ISO3}_rainfall_vs_windspeed.png")

### Exceedance curves and candidate thresholds

The most direct input to trigger design: for a candidate threshold, how often
was it exceeded over the record?

In [ ]:
plot_exceedance(r3, r5, r7, CN, OUT, record_years=config.RECORD_YEARS,
                filename=f"7_{ISO3}_exceedance_thresholds.png")

plot_heatmap(r3, r5, r7, gmax, peak, dates, cats, CN, OUT,
             f"8_{ISO3}_heatmap_all_events.png")

In [ ]:
# Indicative flash-flood thresholds — set per country in config.py
n = len(r3)
n_3d   = int((r3 >= config.THRESH_3D_MM).sum())
n_5d   = int((r5 >= config.THRESH_5D_MM).sum())
n_peak = int((peak >= config.THRESH_PEAK_MMH).sum())

print(f"Events ≥ {config.THRESH_3D_MM} mm in 3 days         : {n_3d}/{n}  ({n_3d/n*100:.0f}%)")
print(f"Events ≥ {config.THRESH_5D_MM} mm in 5 days         : {n_5d}/{n}  ({n_5d/n*100:.0f}%)")
print(f"Events with peak hourly ≥ {config.THRESH_PEAK_MMH} mm/h : {n_peak}/{n}  ({n_peak/n*100:.0f}%)")

print(f"\nHigh peak-hourly events (≥ {config.THRESH_PEAK_MMH} mm/h):")
for i in np.where(peak >= config.THRESH_PEAK_MMH)[0]:
    print(f"  {dates[i]}  {names[i]:<14} {cats[i]:<6} → peak {peak[i]:.1f} mm/h | "
          f"3d {r3[i]:.1f} mm")

### Trend over time

In [ ]:
decade_stats = (
    df_plot.assign(decade=(df_plot["event_datetime"].dt.year // 10) * 10)
    .groupby("decade")[["rain_3d_mm", "rain_5d_mm", "rain_7d_mm", "peak_7d_mmph"]]
    .agg(["mean", "count"])
)
print(decade_stats.to_string())

plot_trend_over_time(years, r3, colors, CN, "3-day rainfall (mm)", OUT,
                     f"9_{ISO3}_trend_over_time.png")

### Summary

In [ ]:
# Generated from the data, so it cannot drift out of step with the outputs.
i_max = int(np.nanargmax(r3))
below_cat3 = [i for i, c in enumerate(cats) if c in ("TS", "Cat 1", "Cat 2")]
wet_below_cat3 = [i for i in below_cat3 if r3[i] > np.median(r3)]

print(f"""
Between {years.min()} and {years.max()}, {CN} experienced {len(r3)} events meeting the
proximity and intensity criteria ({config.IMPACT_RADIUS_KM:.0f} km; ≥{config.HURRICANE_MIN_KT:.0f} kt, or
≥{config.TS_MIN_KT:.0f} kt within {config.IMPACT_RADIUS_KM - config.TS_RADIUS_REDUCTION_KM:.0f} km).

Wettest event: {names[i_max]} ({dates[i_max]}, {cats[i_max]}) — {r3[i_max]:.0f} mm in 3 days,
peak hourly intensity {peak[i_max]:.1f} mm/h.

Of the {len(below_cat3)} events below Category 3, {len(wet_below_cat3)} delivered above-median
3-day rainfall: rainfall hazard is not a simple function of wind category, which
is the case for a rainfall trigger alongside a wind one.

Median 3-day: {np.median(r3):.0f} mm | P75 {np.percentile(r3, 75):.0f} | P90 {np.percentile(r3, 90):.0f}
Median 5-day: {np.median(r5):.0f} mm | P75 {np.percentile(r5, 75):.0f} | P90 {np.percentile(r5, 90):.0f}
Median 7-day: {np.median(r7):.0f} mm | P75 {np.percentile(r7, 75):.0f} | P90 {np.percentile(r7, 90):.0f}
""")

---

**Next:** `03_impact_emdat.ipynb` — join these events to recorded impacts.